# AutoDock Vina Protein-Ligand Docking

![AutoDock Vina Protein-Ligand Docking](https://proto-bio.github.io/proto-assets/images/tool/vina/hero.png)

This notebook redocks imatinib into the c-Abl kinase structure from PDB 1IEP. It defines the crystallographic binding pocket, runs a seeded CPU search, inspects ranked affinity scores, visualizes the best pose, and exports bond-correct SDF coordinates.

In [ ]:
from proto_tools.utils.notebook_docs import display_available_tools, display_api_reference, display_doc_link, display_docs_section, display_overview

display_doc_link("vina")
display_overview("vina")
display_docs_section("vina", "Background")
display_available_tools("vina")

## Redock imatinib into c-Abl

The receptor fixture is the protein from PDB 1IEP. The reference ligand fixture is retained only to demonstrate automatic box construction; the docking ligand itself is supplied as a protonated SMILES string.

In [ ]:
from pathlib import Path

from proto_tools import (
    Structure,
    VinaDockingConfig,
    VinaReferenceLigandBox,
    run_vina_docking,
)
from proto_tools.tools.molecular_docking.vina.vina_docking import (
    example_input,
    example_reference_ligand,
)

display_api_reference("vina", "input", "run_vina_docking")
display_api_reference("vina", "config", "run_vina_docking")
display_api_reference("vina", "output", "run_vina_docking")

In [ ]:
inputs = example_input()

print("Receptor chains:", inputs.receptor.get_chain_ids())
print("Ligand SMILES:", inputs.ligand.smiles)
print("Explicit search box:", inputs.search_box)

reference_box = VinaReferenceLigandBox(reference_ligand=example_reference_ligand(), padding=4.0)
print("Reference-derived box:", reference_box.resolve())

In [ ]:
config = VinaDockingConfig(
    exhaustiveness=8,
    num_poses=5,
    energy_range=4.0,
    cpu=1,
    seed=7,
)

result = run_vina_docking(inputs, config)
result

In [ ]:
import pandas as pd

pd.DataFrame(
    [
        {
            "rank": pose.rank,
            "affinity_kcal_mol": pose.metrics.affinity,
            "rmsd_from_best_lower_bound_angstrom": pose.metrics.rmsd_lower_bound,
            "rmsd_from_best_upper_bound_angstrom": pose.metrics.rmsd_upper_bound,
        }
        for pose in result.poses
    ]
)

## Inspect and export the top-ranked pose

In [ ]:
import py3Dmol

view = py3Dmol.view(width=900, height=600)
view.addModel(inputs.receptor.structure_pdb, "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "spectrum"}})
view.addModel(result.poses[0].sdf, "sdf")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})
view.zoomTo({"model": 1})
view.show()

In [ ]:
output_dir = Path("example_output")
result.export("imatinib_poses", output_dir, file_format="sdf")
result.export("imatinib_scores", output_dir, file_format="csv")
print(output_dir.resolve())

## Fetch the receptor from the PDB

`Structure.from_rcsb` retrieves a deposited entry directly, which is the usual starting point outside this notebook. A deposited entry also carries crystallographic waters and its co-crystallised ligand, so a fetched structure must be reduced to a single cleaned chain before docking: Meeko rejects residues it cannot parameterize unless `allow_bad_residues=True`, which deletes them silently. The bundled fixture used above ships pre-cleaned for that reason, and redocking imatinib into a pocket that still contains imatinib would not be a meaningful validation.

In [ ]:
deposited = Structure.from_rcsb("1IEP")

print("Chains:", deposited.get_chain_ids())
print("Non-polymer ligands in chain A:", deposited.get_chain_ligands("A"))
print("Waters in chain A:", len(deposited.get_chain_waters("A")))

receptor_chain_a = deposited.select_chain("A")
receptor_chain_a